In [7]:
import os, subprocess, shutil
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
print("winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("hadoop.dll:", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot" 
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ";" + os.environ.get("PATH", "")
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("java in PATH:", shutil.which("java"))
print(subprocess.run(["java","-version"], capture_output=True, text=True).stderr)

winutils: True
hadoop.dll: True
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot
java in PATH: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot\bin\java.EXE
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment Temurin-17.0.18+8 (build 17.0.18+8)
OpenJDK 64-Bit Server VM Temurin-17.0.18+8 (build 17.0.18+8, mixed mode, sharing)



In [8]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

import pyspark.pandas as ps
ps.set_option("compute.fail_on_ansi_mode", False)

print("ansi =", spark.conf.get("spark.sql.ansi.enabled"))

ansi = false


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyspark.pandas as ps
from pyspark.sql.functions import *
from datetime import date, timedelta

#### Fonction de chargement des données vers un spark pandas dataframe

In [10]:
def readFile(path, indexCol):
    return ps.read_csv(path, sep=';', index_col=indexCol)

#### Fonctions d'homogénéisation de la langue des fonctions et missions

In [11]:
def clean_langue_fonction(psdf, site):
    match site:
        case "BERLIN":
            map_fonctions = {
                "Ökonom": "Economist",
                "Führungskraft": "Business Executive",
                "Personalleiter": "HRD",
                "Computeringenieur": "Computer Engineer",
                "Dateningenieur": "Data Engineer",
            }
            psdf["FONCTION_PERSONNEL"] = psdf["FONCTION_PERSONNEL"].replace(map_fonctions)
        case "PARIS":
            map_fonctions = {
                "Ingénieur Informaticien": "Computer Engineer",
                "Ingénieur Data": "Data Engineer",
                "Economiste": "Economist",
                "DRH": "HRD",
                "Cadre": "Business Executive",
            }
            psdf["FONCTION_PERSONNEL"] = psdf["FONCTION_PERSONNEL"].replace(map_fonctions)


In [12]:
def clean_langue_mission(psdf, site):
    match site:
        case "BERLIN":
            map_type_mission = {
                "Geschäftstreffen": "Business Meeting",
                "Konferenz": "Conference",
                "Schulung": "Vocational Training",
                "Meeting": "Team Meeting",
                "Entwicklung": "Development",
            }
            psdf["TYPE_MISSION"] = psdf["TYPE_MISSION"].replace(map_type_mission)
        case "PARIS":
            map_type_mission = {
                "Conférence": "Conference",
                "Formation": "Vocational Training",
                "Réunion": "Team Meeting",
                "Rencontre entreprises": "Business Meeting",
                "Développement": "Development",
            }
            psdf["TYPE_MISSION"] = psdf["TYPE_MISSION"].replace(map_type_mission)

#### Chargement et union des données du personnel

In [ ]:
sites = ["PARIS", "BERLIN", "LONDON", "NEWYORK", "SHANGHAI", "LOSANGELES"]
base_path = Path("data/BDD_BGES")

personnel_dfs = []

# On charge la liste du personnel de chaque site dans un dataframe
for site in sites:
    file_path = base_path / f"BDD_BGES_{site}" / f"PERSONNEL_{site}.txt"
    df = pd.read_csv(str(file_path), sep=';', index_col="ID_PERSONNEL")
    clean_langue_fonction(df, site) 
    
    personnel_dfs.append(df)

# On combine les dataframes 
personnel_df = pd.concat(personnel_dfs)

# On extrait le site depuis l'index (ID_PERSONNEL)
personnel_df['SITE'] = personnel_df.index.str.split('_').str[1]

# On sélectionne uniquement les colonnes nécessaires
personnel_final_df = personnel_df[['FONCTION_PERSONNEL', 'SITE']]

personnel_final_df

,FONCTION_PERSONNEL,SITE
ID_PERSONNEL,,
KeyPers_Paris_1230000,Business Executive,Paris
KeyPers_Paris_1230001,HRD,Paris
KeyPers_Paris_1230002,Data Engineer,Paris
KeyPers_Paris_1230003,Computer Engineer,Paris
KeyPers_Paris_1230004,Computer Engineer,Paris
...,...,...
KeyPers_LA_1233022,Business Executive,LA
KeyPers_LA_1233023,Computer Engineer,LA
KeyPers_LA_1233024,Computer Engineer,LA


#### Traitement journalier des données de matériel informatique

In [ ]:
# Définir la plage de dates à traiter
start_date = date(2026, 4, 29)
end_date = date(2026, 5, 25) 

# Liste pour stocker les dataframes de chaque jour
all_materiel_dfs = []

# Boucle sur chaque jour
for single_date in pd.date_range(start_date, end_date):
    day_str = single_date.strftime("%Y%m%d")
    
    # Liste pour stocker les dataframes de matériel de la journée
    materiel_day_dfs = []
    
    # Boucle sur chaque site
    for site in sites:
        file_path = base_path / f"BDD_BGES_{site}" / f"BDD_BGES_{site}_INFORMATIQUE" / f"MATERIEL_INFORMATIQUE_{day_str}.txt"
        
        # Vérifier si le fichier existe avant de le lire
        if file_path.exists():
            df = pd.read_csv(str(file_path), sep=';', engine='python')
            df['DATE'] = single_date
            materiel_day_dfs.append(df)
            
    # Si des fichiers ont été trouvés pour ce jour
    if materiel_day_dfs:
        # Combiner les données de tous les sites pour la journée et l'ajouter à la liste globale
        all_materiel_dfs.append(pd.concat(materiel_day_dfs, ignore_index=True))

# Après la boucle, vérifier si on a collecté des données
if all_materiel_dfs:
    # Combiner les données de tous les jours en un seul DataFrame
    materiel_total_df = pd.concat(all_materiel_dfs, ignore_index=True)
    display(materiel_total_df)
else:
    print("Aucun fichier de matériel trouvé pour la période spécifiée.")


,ID_MATERIELINFO,ID_PERSONNEL,NOM_PERSONNEL,PRENOM_PERSONNEL,DATE_ACHAT,TYPE,MODELE,DATE
0,Paris_MATERIEL_INFO_202604290,KeyPers_Paris_1232362,Name2362,FistName2362,2026-04-29 08:17:31,PC fixe sans ecran,Z,2026-04-29
1,Paris_MATERIEL_INFO_202604291,KeyPers_Paris_1232165,Name2165,FistName2165,2026-04-29 09:42:55,imprimante,Laser A3 (>100kg),2026-04-29
2,Paris_MATERIEL_INFO_202604292,KeyPers_Paris_1231951,Name1951,FistName1951,2026-04-29 13:58:12,PC fixe sans ecran,Precision tower 3xxx,2026-04-29
3,Paris_MATERIEL_INFO_202604293,KeyPers_Paris_1230614,Name614,FistName614,2026-04-29 13:19:31,Telephone IP,,2026-04-29
4,Paris_MATERIEL_INFO_202604294,KeyPers_Paris_1232952,Name2952,FistName2952,2026-04-29 13:55:41,,modèle par défaut,2026-04-29
...,...,...,...,...,...,...,...,...
1098,LA_MATERIEL_INFO_202605257,KeyPers_LA_1230186,Name186,FistName186,2026-05-25 12:51:11,PC fixe sans ecran,"Prodesk (Tower, SFF)",2026-05-25
1099,LA_MATERIEL_INFO_202605258,KeyPers_LA_1232199,Name2199,FistName2199,2026-05-25 11:42:07,Disque dur,modèle par défaut,2026-05-25
1100,LA_MATERIEL_INFO_202605259,KeyPers_LA_1230879,Name879,FistName879,2026-05-25 16:40:33,PC fixe sans ecran,Rasberry pi 4,2026-05-25
1101,LA_MATERIEL_INFO_2026052510,KeyPers_LA_1231490,Name1490,FistName1490,2026-05-25 13:30:40,PC fixe sans ecran,"EliteDesk (Tower, SFF, One)",2026-05-25
